# Week 8: The Complete Training Loop

## Recap

Over the past four weeks we have built up every piece:

- **Week 2-3:** Prediction = `ŷ = w·z + b`
- **Week 4-5:** Gradient = `2(ŷ-y)×z`, Update = `w = w - lr×gradient`
- **Week 6-7:** Multiple weights, simultaneous updates, learning rate effects

Today we assemble all of these into the **complete training loop** — the actual algorithm that trains machine learning models from scratch.

---

## The Complete Algorithm

```
─────────────────────────────────────────────────────────────────────
INITIALIZE:
    w = small random values (e.g., from N(0, 0.01))
    b = 0

FOR epoch = 1 to max_epochs:

    FOR each batch of training examples:
        1. ŷ = Z · w + b            (predict: matrix multiply all examples)
        2. error = ŷ - y            (compute errors for all examples)
        3. grad_w = (2/N) × Z^T · error   (gradient for each weight, averaged)
        4. grad_b = (2/N) × sum(error)    (gradient for bias)

    w = w - lr × grad_w             (update all weights simultaneously)
    b = b - lr × grad_b             (update bias)

    loss = mean((ŷ - y)²)           (record training loss)

    IF max(|grad_w|) < threshold:   (convergence check)
        STOP

DONE — w and b are now trained.
─────────────────────────────────────────────────────────────────────
```

---

## Three Datasets

We will train on three progressively complex datasets:

1. **Loan Approval** — Binary classification, 2 features, 50 samples
2. **House Price Regression** — Continuous output, 3 features, 80 training + 20 test samples
3. **Student Grade Prediction** — 3 features, 60 samples, learning rate comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({
    'figure.facecolor': '#f9f9f9',
    'axes.facecolor':   '#f9f9f9',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
})

np.random.seed(2024)

# ─── Core helper: z-score normalisation ──────────────────────────────────────
def zscore_normalize(X):
    """Returns (X_norm, mean, std) so we can invert later."""
    mu  = X.mean(axis=0)
    sig = X.std(axis=0) + 1e-8
    return (X - mu) / sig, mu, sig

# ─── Core training loop function ─────────────────────────────────────────────
def train(Z, y, lr=0.1, max_epochs=200, convergence_threshold=1e-5, verbose=True):
    """
    Full-batch gradient descent training loop.
    Returns (w, b, loss_history)
    """
    n_features = Z.shape[1]
    N          = len(y)

    # Initialise: small random weights, b=0
    w = np.random.randn(n_features) * 0.01
    b = 0.0

    loss_history   = []
    weight_history = [w.copy()]

    for epoch in range(max_epochs):
        # Step 1: Predict
        y_hat = Z @ w + b

        # Step 2: Error
        error = y_hat - y

        # Step 3: Gradients
        grad_w = (2 / N) * (Z.T @ error)
        grad_b = (2 / N) * np.sum(error)

        # Step 4: Loss
        loss = np.mean(error ** 2)
        loss_history.append(loss)

        # Step 5: Update
        w = w - lr * grad_w
        b = b - lr * grad_b

        weight_history.append(w.copy())

        # Convergence check
        if np.max(np.abs(grad_w)) < convergence_threshold:
            if verbose:
                print(f"  Converged at epoch {epoch+1} (max gradient = {np.max(np.abs(grad_w)):.2e})")
            break

    return w, b, loss_history, weight_history

print("Helper functions loaded. Starting training experiments!")

---
# Dataset 1: Loan Approval (Binary Classification)

## Data Generation

We create 50 synthetic applicants. The true rule (which the model must discover):
- Approved (y=1) if: `age_score × 0.6 + income_score × 0.4 + noise > 0`
- Rejected (y=0) otherwise

Features (before normalization):
- `age`: roughly 25-65 years
- `income`: roughly 30k-120k

In [ ]:
# ─── Dataset 1: Loan Approval ─────────────────────────────────────────────────
np.random.seed(2024)

N1   = 50
age  = np.random.uniform(25, 65, N1)
inc  = np.random.uniform(30_000, 120_000, N1)

# Combine raw features into matrix
X1_raw = np.column_stack([age, inc])

# Normalise
Z1, mu1, sig1 = zscore_normalize(X1_raw)

# True labels: weighted combination of z-scores + noise
true_signal = 0.6 * Z1[:, 0] + 0.4 * Z1[:, 1] + np.random.randn(N1) * 0.5
y1          = (true_signal > 0).astype(float)

print("=" * 60)
print("DATASET 1: LOAN APPROVAL")
print("=" * 60)
print(f"  Samples: {N1}")
print(f"  Features: age, income  (z-score normalised)")
print(f"  Approved (y=1): {int(y1.sum())} | Rejected (y=0): {int((1-y1).sum())}")
print()
print(f"  Z1 shape: {Z1.shape}")
print(f"  Z1 mean:  {Z1.mean(axis=0)}  (should be ~0)")
print(f"  Z1 std:   {Z1.std(axis=0)}   (should be ~1)")
print()
print("  First 5 samples:")
print(f"  {'z_age':>8} {'z_income':>10} {'y':>5}")
for i in range(5):
    print(f"  {Z1[i,0]:>8.3f} {Z1[i,1]:>10.3f} {y1[i]:>5.0f}")

In [ ]:
# ─── Train on Dataset 1 ───────────────────────────────────────────────────────
print("Training on Loan Approval dataset...")
print()

w_init_1 = np.random.randn(2) * 0.01
print(f"  Initial weights: w_age={w_init_1[0]:.6f}, w_income={w_init_1[1]:.6f}")
print()

np.random.seed(2024)
w1, b1, loss1, wh1 = train(Z1, y1, lr=0.1, max_epochs=200, verbose=True)

print()
print(f"  Initial weights: w_age≈{wh1[0][0]:.6f},  w_income≈{wh1[0][1]:.6f}")
print(f"  Epoch 10 weights: w_age={wh1[10][0]:.5f}, w_income={wh1[10][1]:.5f}")
print(f"  Epoch 50 weights: w_age={wh1[min(50,len(wh1)-1)][0]:.5f}, w_income={wh1[min(50,len(wh1)-1)][1]:.5f}")
print(f"  Final weights:   w_age={w1[0]:.5f},  w_income={w1[1]:.5f},  b={b1:.5f}")
print()
print("  Interpretation:")
print(f"    w_age={w1[0]:.3f}    → positive: older applicants more likely approved")
print(f"    w_income={w1[1]:.3f} → positive: higher income more likely approved")
print(f"    (True generating weights: w_age=0.6, w_income=0.4)")

# Predictions on 5 test examples
print()
print("  Predictions on first 5 test examples:")
print(f"  {'z_age':>8} {'z_income':>10} {'y_true':>8} {'ŷ':>8} {'correct':>9}")
for i in range(5):
    y_hat_i = np.dot(Z1[i], w1) + b1
    pred_i  = round(y_hat_i)
    ok      = "✓" if pred_i == y1[i] else "✗"
    print(f"  {Z1[i,0]:>8.3f} {Z1[i,1]:>10.3f} {y1[i]:>8.0f} {y_hat_i:>8.4f}   {ok}")

In [ ]:
# ─── Visualise Dataset 1 results ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Dataset 1: Loan Approval — Training Results', fontsize=13, fontweight='bold')

# Loss curve
axes[0].plot(loss1, 'o-', color='steelblue', lw=2, ms=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean Squared Loss')
axes[0].set_title('Loss Curve')
axes[0].fill_between(range(len(loss1)), loss1, alpha=0.15, color='steelblue')

# Weight evolution
wh1_arr = np.array(wh1)
axes[1].plot(wh1_arr[:, 0], label='w_age',    color='crimson',    lw=2)
axes[1].plot(wh1_arr[:, 1], label='w_income', color='darkorange', lw=2)
axes[1].axhline(0.6, color='crimson',    lw=1.5, linestyle='--', alpha=0.5, label='True w_age=0.6*')
axes[1].axhline(0.4, color='darkorange', lw=1.5, linestyle='--', alpha=0.5, label='True w_income=0.4*')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Weight value')
axes[1].set_title('Weight Convergence')
axes[1].legend(fontsize=8)

# Final predictions vs true labels
y_hat_all1 = Z1 @ w1 + b1
colors1    = ['crimson' if y == 1 else 'steelblue' for y in y1]
axes[2].scatter(range(N1), y_hat_all1, c=colors1, alpha=0.7, s=50, edgecolors='black', lw=0.5)
axes[2].axhline(0.5, color='green', lw=1.5, linestyle='--', label='Decision boundary')
axes[2].set_xlabel('Sample index')
axes[2].set_ylabel('Prediction ŷ')
axes[2].set_title('Final Predictions\n(red=approved, blue=rejected true labels)')
axes[2].legend(fontsize=9)

patch_a = plt.matplotlib.patches.Patch(color='crimson',   label='y=1 (approved)')
patch_r = plt.matplotlib.patches.Patch(color='steelblue', label='y=0 (rejected)')
axes[2].legend(handles=[patch_a, patch_r] + axes[2].get_legend_handles_labels()[0][-1:], fontsize=8)

plt.tight_layout()
plt.show()

---
# Dataset 2: House Price Regression

## Data Generation

80 training samples, 20 test samples. Three features:
- `size_m2`: floor area in square metres (40–250 m²)
- `n_rooms`: number of rooms (1–6)
- `location_score`: location quality rating (1–10)

True generating formula (what the model must recover):  
`price_norm = 0.5×z_size + 0.3×z_rooms + 0.2×z_location + noise`

After training we evaluate MSE on the held-out 20 test samples.

In [ ]:
# ─── Dataset 2: House Price ────────────────────────────────────────────────────
np.random.seed(2024)

N2_total = 100

size_m2    = np.random.uniform(40, 250, N2_total)
n_rooms    = np.random.randint(1, 7, N2_total).astype(float)
loc_score  = np.random.uniform(1, 10, N2_total)

X2_raw = np.column_stack([size_m2, n_rooms, loc_score])

# Normalize ALL data together, then split — prevents data leakage in a simple way
Z2_all, mu2, sig2 = zscore_normalize(X2_raw)

# True output
y2_all = (0.5 * Z2_all[:, 0] +
          0.3 * Z2_all[:, 1] +
          0.2 * Z2_all[:, 2] +
          np.random.randn(N2_total) * 0.3)

# Split: first 80 train, last 20 test
Z2_train, y2_train = Z2_all[:80], y2_all[:80]
Z2_test,  y2_test  = Z2_all[80:], y2_all[80:]

print("=" * 60)
print("DATASET 2: HOUSE PRICE REGRESSION")
print("=" * 60)
print(f"  Training samples: {len(y2_train)}")
print(f"  Test samples:     {len(y2_test)}")
print(f"  Features: z_size, z_rooms, z_location")
print(f"  Target range: [{y2_all.min():.2f}, {y2_all.max():.2f}]")
print()
print("  First 3 training examples:")
print(f"  {'z_size':>8} {'z_rooms':>9} {'z_loc':>7} {'price':>8}")
for i in range(3):
    print(f"  {Z2_train[i,0]:>8.3f} {Z2_train[i,1]:>9.3f} {Z2_train[i,2]:>7.3f} {y2_train[i]:>8.4f}")

In [ ]:
# ─── Train on Dataset 2 ───────────────────────────────────────────────────────
print("Training on House Price dataset...")
print()

np.random.seed(2024)
w2, b2, loss2, wh2 = train(Z2_train, y2_train, lr=0.05, max_epochs=500, verbose=True)

# Evaluate on test set
y_hat_test2 = Z2_test @ w2 + b2
mse_test2   = np.mean((y_hat_test2 - y2_test) ** 2)
mae_test2   = np.mean(np.abs(y_hat_test2 - y2_test))

print()
print(f"  Final training loss:  {loss2[-1]:.5f}")
print(f"  Test MSE:             {mse_test2:.5f}")
print(f"  Test MAE:             {mae_test2:.5f}")
print()
print("  Learned vs True weights:")
print(f"  {'Feature':<12} {'Learned':>10} {'True':>8}")
print(f"  {'-'*32}")
print(f"  {'w_size':<12} {w2[0]:>10.5f} {0.5:>8.3f}")
print(f"  {'w_rooms':<12} {w2[1]:>10.5f} {0.3:>8.3f}")
print(f"  {'w_location':<12} {w2[2]:>10.5f} {0.2:>8.3f}")
print(f"  {'bias':<12} {b2:>10.5f} {0.0:>8.3f}")

In [ ]:
# ─── Visualise Dataset 2 results ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Dataset 2: House Price Regression — Full Results', fontsize=13, fontweight='bold')

# 1. Training loss
axes[0, 0].semilogy(loss2, color='steelblue', lw=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (log scale)')
axes[0, 0].set_title('Training Loss Curve')

# 2. Weight convergence
wh2_arr = np.array(wh2)
colors_w = ['steelblue', 'darkorange', 'green']
labels_w = ['w_size (true=0.5)', 'w_rooms (true=0.3)', 'w_location (true=0.2)']
for j, (c, l, tv) in enumerate(zip(colors_w, labels_w, [0.5, 0.3, 0.2])):
    axes[0, 1].plot(wh2_arr[:, j], color=c, lw=2, label=f'Learned: {l}')
    axes[0, 1].axhline(tv, color=c, lw=1.5, linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Weight value')
axes[0, 1].set_title('Weight Convergence (dashed = true)')
axes[0, 1].legend(fontsize=8)

# 3. Predicted vs Actual on TRAINING set
y_hat_train2 = Z2_train @ w2 + b2
axes[1, 0].scatter(y2_train, y_hat_train2, color='steelblue', alpha=0.6, s=40, label='Training')
lo2 = min(y2_train.min(), y_hat_train2.min()) - 0.1
hi2 = max(y2_train.max(), y_hat_train2.max()) + 0.1
axes[1, 0].plot([lo2, hi2], [lo2, hi2], 'r--', lw=1.5, label='Perfect fit')
axes[1, 0].set_xlabel('Actual price (normalised)')
axes[1, 0].set_ylabel('Predicted price')
axes[1, 0].set_title(f'Predicted vs Actual (Train)\nMSE={loss2[-1]:.4f}')
axes[1, 0].legend(fontsize=9)

# 4. Predicted vs Actual on TEST set
axes[1, 1].scatter(y2_test, y_hat_test2, color='crimson', alpha=0.7, s=60, label='Test')
lo2t = min(y2_test.min(), y_hat_test2.min()) - 0.1
hi2t = max(y2_test.max(), y_hat_test2.max()) + 0.1
axes[1, 1].plot([lo2t, hi2t], [lo2t, hi2t], 'b--', lw=1.5, label='Perfect fit')
axes[1, 1].set_xlabel('Actual price (normalised)')
axes[1, 1].set_ylabel('Predicted price')
axes[1, 1].set_title(f'Predicted vs Actual (Test, n=20)\nMSE={mse_test2:.4f}  MAE={mae_test2:.4f}')
axes[1, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()

**Reading the plots:**

- **Top-left (loss):** Drops rapidly in early epochs as the model makes large corrections, then flattens as weights approach their optimal values.

- **Top-right (weights):** Each learned weight converges toward the true generating weight. Small differences remain due to noise in the training data.

- **Bottom-left/right (scatter):** Points close to the diagonal line = good predictions. The cloud of points should be tight around the diagonal. The test set results are similar to training — no overfitting.

---
# Dataset 3: Student Grade Prediction — Learning Rate Comparison

## Data Generation

60 students, 3 features:
- `z_study_hours`: standardised weekly study hours
- `z_attendance`: standardised attendance percentage
- `z_sleep`: standardised average nightly sleep

True generating formula:  
`grade_norm = 0.5×z_study + 0.3×z_attend + 0.2×z_sleep + noise`

We train the same dataset with **four learning rates** and compare convergence.

In [ ]:
# ─── Dataset 3: Student Grades ────────────────────────────────────────────────
np.random.seed(2024)

N3 = 60

study_hrs  = np.random.uniform(2, 30, N3)
attendance = np.random.uniform(50, 100, N3)
sleep_hrs  = np.random.uniform(4, 10, N3)

X3_raw = np.column_stack([study_hrs, attendance, sleep_hrs])
Z3, mu3, sig3 = zscore_normalize(X3_raw)

y3 = (0.5 * Z3[:, 0] +
      0.3 * Z3[:, 1] +
      0.2 * Z3[:, 2] +
      np.random.randn(N3) * 0.25)

print("=" * 60)
print("DATASET 3: STUDENT GRADE PREDICTION")
print("=" * 60)
print(f"  Samples: {N3}")
print(f"  Features: z_study, z_attendance, z_sleep")
print(f"  Target range: [{y3.min():.2f}, {y3.max():.2f}]")
print()
print("  Four learning rates will be compared: 0.001, 0.1, 0.5, 2.0")

In [ ]:
# ─── Train with four learning rates ──────────────────────────────────────────
lr_experiments = [
    (0.001, 'lr=0.001 (too small)', '#aec7e8'),
    (0.1,   'lr=0.1   (good)',      '#1f77b4'),
    (0.5,   'lr=0.5   (large)',     '#ff7f0e'),
    (2.0,   'lr=2.0   (diverges)',  '#d62728'),
]

max_ep = 300
lr_results = {}

for lr_val, label, color in lr_experiments:
    np.random.seed(42)   # same starting weights each time for fair comparison
    try:
        w_r, b_r, losses_r, wh_r = train(Z3, y3,
                                          lr=lr_val,
                                          max_epochs=max_ep,
                                          convergence_threshold=1e-6,
                                          verbose=False)
        # Clip diverging losses for display
        losses_clipped = [min(l, 1e4) for l in losses_r]
        status = f"Final loss: {losses_r[-1]:.5f}"
    except Exception as e:
        losses_clipped = [1e4] * max_ep
        w_r = np.zeros(3)
        b_r = 0.0
        status = f"ERROR: {e}"

    lr_results[label] = {
        'losses': losses_clipped,
        'color': color,
        'status': status,
        'w': w_r,
        'b': b_r,
    }
    print(f"  {label:30s}: {status}")

In [ ]:
# ─── Learning rate comparison plots ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset 3: Student Grades — Learning Rate Comparison (300 epochs)', fontsize=13, fontweight='bold')

iters_3 = range(max_ep)

for label, res in lr_results.items():
    losses_to_plot = res['losses'][:max_ep]
    axes[0].plot(iters_3[:len(losses_to_plot)], losses_to_plot,
                 lw=2, color=res['color'], label=label)
    safe_losses = [max(l, 1e-12) for l in losses_to_plot]
    axes[1].semilogy(iters_3[:len(safe_losses)], safe_losses,
                     lw=2, color=res['color'], label=label)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Linear Scale\n(diverging lr dominates)')
axes[0].set_ylim(-0.1, 5)
axes[0].legend(fontsize=9)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Log Scale\n(clearly separates convergence rates)')
axes[1].legend(fontsize=9)
axes[1].axhline(0.1, color='gray', lw=1, linestyle=':', alpha=0.6)
axes[1].text(10, 0.12, 'Reasonable loss threshold', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# ─── Weight trajectories for each lr ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset 3: Student Grades — Final Predictions vs Actual (good lr=0.1)', fontsize=12, fontweight='bold')

# Use lr=0.1 results for final evaluation
label_good = 'lr=0.1   (good)'
w_good     = lr_results[label_good]['w']
b_good     = lr_results[label_good]['b']
y_hat_3    = Z3 @ w_good + b_good

# Sort by actual grade for a cleaner comparison plot
sort_idx = np.argsort(y3)

axes[0].plot(y3[sort_idx],     'o-', color='steelblue', lw=1.5, ms=4, label='Actual grade')
axes[0].plot(y_hat_3[sort_idx],'s--', color='crimson',   lw=1.5, ms=4, label='Predicted grade')
axes[0].set_xlabel('Student (sorted by actual grade)')
axes[0].set_ylabel('Normalised grade')
axes[0].set_title('Actual vs Predicted — Sorted View')
axes[0].legend(fontsize=9)

# Scatter: predicted vs actual
axes[1].scatter(y3, y_hat_3, color='purple', alpha=0.6, s=60, edgecolors='black', lw=0.5)
lo3 = min(y3.min(), y_hat_3.min()) - 0.1
hi3 = max(y3.max(), y_hat_3.max()) + 0.1
axes[1].plot([lo3, hi3], [lo3, hi3], 'r--', lw=1.5, label='Perfect fit')
mse3_good = np.mean((y_hat_3 - y3)**2)
axes[1].set_xlabel('Actual grade')
axes[1].set_ylabel('Predicted grade')
axes[1].set_title(f'Scatter Plot (lr=0.1)\nMSE={mse3_good:.4f}')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"  Final weights (lr=0.1):")
print(f"    w_study    = {w_good[0]:.4f}  (true = 0.5)")
print(f"    w_attend   = {w_good[1]:.4f}  (true = 0.3)")
print(f"    w_sleep    = {w_good[2]:.4f}  (true = 0.2)")
print(f"    bias b     = {b_good:.4f}")
print(f"  Training MSE: {mse3_good:.4f}")

---
# Putting It All Together: The Step-By-Step Training Walkthrough

Let's do one final, detailed walkthrough of the very first epoch of training on Dataset 1 — showing every single calculation transparently, as if we were doing it by hand on a whiteboard.

In [ ]:
# ─── Manual walkthrough: first epoch on Dataset 1 (first 5 samples only) ─────
print("=" * 68)
print("TRANSPARENT WALKTHROUGH: Epoch 1 on 5 Loan Approval Samples")
print("=" * 68)

np.random.seed(2024)
w_walk = np.array([0.005, -0.003])    # small random init
b_walk = 0.0
lr_w   = 0.1

# Use first 5 samples from Dataset 1
Z_walk = Z1[:5]
y_walk = y1[:5]

print(f"  Initial weights: w=[{w_walk[0]:.4f}, {w_walk[1]:.4f}], b={b_walk}")
print(f"  Learning rate: {lr_w}")
print()
print("  STEP 1: Predict (ŷ = Z·w + b)")
y_hat_walk = Z_walk @ w_walk + b_walk
for i in range(5):
    print(f"    Sample {i+1}: ŷ = {w_walk[0]:.4f}×{Z_walk[i,0]:+.3f} + {w_walk[1]:.4f}×{Z_walk[i,1]:+.3f} + {b_walk} = {y_hat_walk[i]:.5f}")

print()
print("  STEP 2: Compute errors (error = ŷ - y)")
errors_walk = y_hat_walk - y_walk
for i in range(5):
    print(f"    Sample {i+1}: {y_hat_walk[i]:.5f} - {y_walk[i]:.0f} = {errors_walk[i]:+.5f}")

print()
print("  STEP 3: Compute losses")
losses_walk = errors_walk ** 2
for i in range(5):
    print(f"    Sample {i+1}: ({errors_walk[i]:+.5f})² = {losses_walk[i]:.6f}")
print(f"    Mean loss: {losses_walk.mean():.6f}")

print()
print("  STEP 4: Compute gradients (averaged over 5 samples)")
N_w  = len(y_walk)
gw   = (2 / N_w) * (Z_walk.T @ errors_walk)
gb   = (2 / N_w) * np.sum(errors_walk)
print(f"    grad_w_age    = (2/5) × Σ(error × z_age)    = {gw[0]:+.6f}")
print(f"    grad_w_income = (2/5) × Σ(error × z_income) = {gw[1]:+.6f}")
print(f"    grad_b        = (2/5) × Σ(error)            = {gb:+.6f}")

print()
print("  STEP 5: Update weights (simultaneously)")
w_new_walk = w_walk - lr_w * gw
b_new_walk = b_walk - lr_w * gb
print(f"    w_age:    {w_walk[0]:+.6f} - {lr_w}×({gw[0]:+.6f}) = {w_new_walk[0]:+.6f}")
print(f"    w_income: {w_walk[1]:+.6f} - {lr_w}×({gw[1]:+.6f}) = {w_new_walk[1]:+.6f}")
print(f"    b:        {b_walk:+.6f} - {lr_w}×({gb:+.6f}) = {b_new_walk:+.6f}")

print()
print("  VERIFY: New predictions with updated weights")
y_hat_new_walk = Z_walk @ w_new_walk + b_new_walk
new_loss = np.mean((y_hat_new_walk - y_walk)**2)
print(f"    New mean loss: {new_loss:.6f}  vs  old: {losses_walk.mean():.6f}")
print(f"    Improvement: {'YES ✓' if new_loss < losses_walk.mean() else 'NO ✗'}")

---
# The Pre-Training Checklist

Before you train any model, run through this checklist. These are the most common mistakes beginners make.

## 1. Normalize All Features (z-score)

**Why:** If feature A is in the range [0,1] and feature B is in the range [10,000 - 100,000], the gradient for B's weight will be 100,000× larger. Training becomes unstable and slow.

**How:** For each feature column: `z = (x - mean) / std`

**Result:** All features have mean ≈ 0 and std ≈ 1. Gradients are comparable in magnitude.


In [ ]:
# ─── Checklist Item 1: Demonstrate why normalisation matters ─────────────────
np.random.seed(42)

# Raw data — very different scales
X_raw_check = np.column_stack([
    np.random.uniform(25, 65, 50),               # age: 25-65
    np.random.uniform(30_000, 120_000, 50),      # income: 30k-120k
])
y_check = np.random.randint(0, 2, 50).astype(float)

# Train WITHOUT normalisation
w_no, b_no, loss_no, _ = train(X_raw_check, y_check, lr=0.00001,
                                 max_epochs=200, verbose=False)

# Train WITH normalisation
X_norm_check, _, _ = zscore_normalize(X_raw_check)
w_ok, b_ok, loss_ok, _ = train(X_norm_check, y_check, lr=0.1,
                                 max_epochs=200, verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Checklist #1: Why Feature Normalization Matters', fontsize=12, fontweight='bold')

axes[0].plot(loss_no, color='crimson', lw=2, label='Without normalisation\n(lr=0.00001 — had to be tiny!)')
axes[0].plot(loss_ok, color='steelblue', lw=2, label='With normalisation\n(lr=0.1 — works great)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Comparison')
axes[0].legend(fontsize=9)

# Show raw vs normalised feature distributions
axes[1].boxplot([X_raw_check[:, 0], X_raw_check[:, 1] / 1000,
                 X_norm_check[:, 0], X_norm_check[:, 1]],
                labels=['age\n(raw)', 'income/1k\n(raw)', 'z_age\n(norm)', 'z_income\n(norm)'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', lw=2))
axes[1].axhline(0, color='gray', lw=1, linestyle='--', alpha=0.5)
axes[1].set_title('Raw vs Normalised Feature Distributions\n(normalised: centred at 0, same scale)')
axes[1].set_ylabel('Value')

plt.tight_layout()
plt.show()

print(f"  Without normalisation — final loss: {loss_no[-1]:.4f} (needed lr=0.00001)")
print(f"  With normalisation    — final loss: {loss_ok[-1]:.4f} (lr=0.1 works fine)")

## 2. Initialize Weights to Small Random Values

**Why not zero?** If all weights start at zero, all gradients will be identical (symmetry problem). The model can't learn distinct patterns for different features.

**How:** `w = np.random.randn(n_features) * 0.01`  
Small values (not 1.0, not 100.0) prevent exploding gradients on the first step.

## 3. Choose Learning Rate

**Rule of thumb:** Start with `0.01` or `0.1`. Then look at the loss curve:
- Loss drops steadily → good
- Loss barely moves → increase lr by 10×
- Loss explodes → decrease lr by 10×

## 4. Set Max Epochs

- Start with `100-500` for small datasets
- Increase if loss is still decreasing when training stops
- Decrease if training takes too long and loss has plateaued

## 5. Monitor the Loss Curve

The loss curve tells you everything:

In [ ]:
# ─── Loss curve shape gallery ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
fig.suptitle('Checklist #5: What Different Loss Curves Mean', fontsize=12, fontweight='bold')

epochs_gallery = np.arange(100)

# Good convergence
loss_good = 1.0 * np.exp(-0.1 * epochs_gallery) + 0.05
axes[0].plot(epochs_gallery, loss_good, 'steelblue', lw=2)
axes[0].set_title('GOOD\nSmooth decrease to plateau', fontsize=10, color='green')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

# Too slow (lr too small)
loss_slow = 1.0 * np.exp(-0.005 * epochs_gallery) + 0.05
axes[1].plot(epochs_gallery, loss_slow, 'orange', lw=2)
axes[1].set_title('TOO SLOW\nNeeds more epochs or bigger lr', fontsize=10, color='darkorange')
axes[1].set_xlabel('Epoch')

# Oscillating (lr slightly too large)
loss_osc = 0.3 * np.exp(-0.05 * epochs_gallery) * np.abs(np.sin(epochs_gallery * 0.5)) + \
           0.3 * np.exp(-0.04 * epochs_gallery) + 0.05
axes[2].plot(epochs_gallery, loss_osc, 'darkorange', lw=2)
axes[2].set_title('OSCILLATING\nReduce lr slightly', fontsize=10, color='darkorange')
axes[2].set_xlabel('Epoch')

# Diverging
loss_div = np.minimum(1.0 * np.exp(0.08 * epochs_gallery), 50)
axes[3].plot(epochs_gallery, loss_div, 'crimson', lw=2)
axes[3].set_title('DIVERGING\nReduce lr immediately!', fontsize=10, color='red')
axes[3].set_xlabel('Epoch')

for ax in axes:
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

## 6. Stop When Loss Plateaus

**Convergence check:**  
`if max(|grad_w|) < threshold: STOP`

Or simply: if the loss hasn't decreased by more than `ε = 1e-5` in the last 10 epochs, stop.

Continuing past convergence wastes compute but doesn't hurt (with simple linear models). In more complex models it can cause overfitting.

---

# Complete Pre-Training Checklist


In [ ]:
# ─── Display the complete checklist as a final summary ───────────────────────
checklist = [
    ("1", "Normalize features",         "z = (x - mean) / std for every feature column"),
    ("2", "Initialize weights",         "w = np.random.randn(n) * 0.01, b = 0"),
    ("3", "Choose learning rate",        "Start with 0.01 or 0.1; adjust based on loss curve"),
    ("4", "Set max epochs",              "100-500 for small datasets; increase if still improving"),
    ("5", "Monitor loss",               "Should decrease. Flat=too slow. Spiking=too fast. Exploding=diverging"),
    ("6", "Stop at convergence",         "When loss plateaus or max|grad| < 1e-5"),
    ("7", "Evaluate on held-out data",   "Always check test MSE — training loss alone doesn't tell the full story"),
]

print("=" * 72)
print("CHECKLIST: Before You Train Any Model")
print("=" * 72)
for num, title, detail in checklist:
    print(f"  [{num}] {title}")
    print(f"       → {detail}")
    print()

---
# Final Summary

## What We Built

Over the past 5 weeks, you have built a complete linear regression/classification model from scratch — no libraries, no black boxes. Here is the complete picture:

```
DATA PIPELINE:
  Raw data  →  z-score normalise  →  Z matrix (N × F)

FORWARD PASS (prediction):
  ŷ = Z · w + b

LOSS:
  L = mean((ŷ - y)²)       ← mean squared error

BACKWARD PASS (gradients):
  ∂L/∂w = (2/N) × Z^T · (ŷ - y)   ← one number per weight
  ∂L/∂b = (2/N) × sum(ŷ - y)      ← one number for bias

UPDATE (gradient descent):
  w = w - lr × ∂L/∂w
  b = b - lr × ∂L/∂b

REPEAT until converged.
```

## Key Numbers to Remember

| Hyperparameter | Typical starting value | Adjust if... |
|---|---|---|
| Learning rate | `0.01` or `0.1` | Loss explodes → divide by 10; loss flat → multiply by 10 |
| Max epochs | `200` | Still improving at end → double |
| Weight init scale | `0.01` | Gradients explode on epoch 1 → smaller |
| Convergence threshold | `1e-5` | Converges too early → smaller |

## What Comes Next

With this foundation you are ready for:
- **Logistic regression:** Add a sigmoid activation to output probabilities between 0 and 1
- **Neural networks:** Stack multiple layers, each with its own weights and activation
- **Mini-batching:** Split training data into small batches for faster, more stable training
- **Regularization:** Add penalties to prevent overfitting on real-world noisy data

But the core loop — predict, compute error, compute gradient, update — never changes. Every deep learning model, from image classifiers to large language models, is built on exactly what you learned here.